In [ ]:
import pandas as pd
import os

In [ ]:
silver_path = "data/silver"
gold_path = "data/gold"

os.makedirs(silver_path, exist_ok=True)
os.makedirs(gold_path, exist_ok=True)

In [ ]:
print("lendo silver...")
silver_df = pd.read_csv("data/silver/customer_journey_silver.csv")
print(f"print {len(silver_df):,} linhas | {silver_df['SessionID'].nunique():,} sessões | {silver_df['UserID'].nunique():,} usuários")

lendo silver...
print 12,719 linhas | 5,000 sessões | 1,872 usuários


In [ ]:
silver_df.head(5)

,SessionID,UserID,Timestamp,PageType,DeviceType,Country,ReferralSource,TimeOnPage_seconds,ItemsInCart,Purchased,funnel_order
0,session_0,user_2223,2025-01-20 22:53:34,home,Desktop,India,Social Media,55,0,0,1
1,session_1,user_2192,2025-02-26 12:57:10,home,Tablet,Germany,Email,99,0,0,1
2,session_1,user_2192,2025-02-26 12:59:11,product_page,Tablet,Germany,Email,121,0,0,2
3,session_10,user_2357,2025-05-17 22:11:37,home,Tablet,India,Direct,75,0,0,1
4,session_10,user_2357,2025-05-17 22:13:33,product_page,Tablet,India,Direct,116,0,0,2


In [ ]:
dim_session = (
    silver_df.groupby("SessionID", sort=False)
    .agg(
        UserID         = ("UserID",        "first"),
        DeviceType     = ("DeviceType",    "first"),
        Country        = ("Country",       "first"),
        ReferralSource = ("ReferralSource","first"),
    )
    .reset_index()
)

dim_session["SessionKey"] = dim_session.index + 1
dim_session.head(5)

,SessionID,UserID,DeviceType,Country,ReferralSource,SessionKey
0,session_0,user_2223,Desktop,India,Social Media,1
1,session_1,user_2192,Tablet,Germany,Email,2
2,session_10,user_2357,Tablet,India,Direct,3
3,session_100,user_1233,Desktop,USA,Email,4
4,session_1000,user_1551,Mobile,France,Email,5


In [ ]:
dim_page = (
    silver_df[["PageType", "funnel_order"]]
    .drop_duplicates()
    .sort_values("funnel_order")
    .reset_index(drop=True)
)

dim_page["PageID"] = dim_page.index + 1

funnel_labels = {
    1: "Home",
    2: "Product Page",
    3: "Cart",
    4: "Checkout",
    5: "Confirmation",
}

dim_page["funnel_label"] = dim_page["funnel_order"].map(funnel_labels)

dim_page.head(5)

,PageType,funnel_order,PageID,funnel_label
0,home,1,1,Home
1,product_page,2,2,Product Page
2,cart,3,3,Cart
3,checkout,4,4,Checkout
4,confirmation,5,5,Confirmation


In [ ]:
dim_calendar = pd.DataFrame()
dim_calendar["FullDate"] = pd.to_datetime(silver_df["Timestamp"]).drop_duplicates().sort_values(ascending=True).reset_index(drop=True)
dim_calendar["Year"] = dim_calendar["FullDate"].dt.year
dim_calendar["Quarter"] = dim_calendar["FullDate"].dt.quarter
dim_calendar["Month"] = dim_calendar["FullDate"].dt.month
dim_calendar["MonthName"] = dim_calendar["FullDate"].dt.strftime("%B")
dim_calendar["Day"] = dim_calendar["FullDate"].dt.day
dim_calendar["WeekdayNumber"] = dim_calendar["FullDate"].dt.dayofweek   # 0=Mon, 6=Sun
dim_calendar["WeekdayName"] = dim_calendar["FullDate"].dt.strftime("%A")
dim_calendar["DateID"] = dim_calendar.index + 1
dim_calendar.head(5)


,FullDate,Year,Quarter,Month,MonthName,Day,WeekdayNumber,WeekdayName,DateID
0,2025-01-01 00:09:11,2025,1,1,January,1,2,Wednesday,1
1,2025-01-01 00:10:02,2025,1,1,January,1,2,Wednesday,2
2,2025-01-01 00:25:04,2025,1,1,January,1,2,Wednesday,3
3,2025-01-01 00:26:45,2025,1,1,January,1,2,Wednesday,4
4,2025-01-01 00:29:19,2025,1,1,January,1,2,Wednesday,5


In [ ]:
len(dim_calendar)

12713

In [ ]:
fact_table = silver_df.copy()
fact_table["Timestamp"] = fact_table["Timestamp"].astype("datetime64[ns]")

In [ ]:
session_map = dim_session.set_index("SessionID")["SessionKey"]
page_map    = dim_page.set_index("PageType")["PageID"]


In [ ]:
def merge_dim_calendar(df: pd.DataFrame, dimension: pd.DataFrame) -> pd.DataFrame:
  columns = []
  for col in dimension.columns.to_list():
    if col not in ["DateID"]:
      columns.append(col)
  df = pd.merge(df, dimension, left_on="Timestamp", right_on="FullDate", how="left")
  df.drop(columns=columns, inplace=True)
  return df

In [ ]:
fact_table["SessionKey"] = fact_table["SessionID"].map(session_map)
fact_table["PageID"]    = fact_table["PageType"].map(page_map)

fact_table = merge_dim_calendar(fact_table, dim_calendar)

fact_table = fact_table[[
    "SessionKey",
    "PageID",
    "DateID",
    "Timestamp",            # timestamp completo (quando precisar de granularidade hora)
    "TimeOnPage_seconds",
    "ItemsInCart",
    "Purchased",
]].rename(columns={"Timestamp": "EventTimestamp"})

fact_table.head(5)

,SessionKey,PageID,DateID,EventTimestamp,TimeOnPage_seconds,ItemsInCart,Purchased
0,1,1,1111,2025-01-20 22:53:34,55,0,0
1,2,1,3044,2025-02-26 12:57:10,99,0,0
2,2,2,3045,2025-02-26 12:59:11,121,0,0
3,3,1,7122,2025-05-17 22:11:37,75,0,0
4,3,2,7123,2025-05-17 22:13:33,116,0,0


In [ ]:
dim_session.to_csv(f"{gold_path}/dim_session.csv", index=False)
dim_page.to_csv(f"{gold_path}/dim_page.csv", index=False)
dim_calendar.to_csv(f"{gold_path}/dim_calendar.csv", index=False)
fact_table.to_csv(f"{gold_path}/fact_table.csv", index=False)